Reissner Mindlin Plate : Modal Analysis with Fenicsx 
=================

First of all, this work is base on Jeremie Bleyer's work which can be found with this link : [Fenicsx Tour : Reissner-Mindlin plates ](https://bleyerj.github.io/comet-fenicsx/intro/plates/plates.html)

Governor Equation
===============

Generalized strain
---------------

- Courbure : $ \chi = \nabla^S \phi  $
- Déformation de cisaillement : $ \gamma = \nabla u - \phi $

Generalized stresses
----------------

- Moment lié à la courbure : $ \textbf{M} $
- Force de cisaillement : $ \textbf{V} $ 

Formulation Forte 
----------------

 - $ V_{ \beta , \beta } = M \ddot U_3 $
 - $ M_{ \alpha \beta , \beta } - V_\alpha = I \ddot \phi _\alpha $

Implementation Start 
-----------------
Importation des bibliothéque utile

In [41]:
import numpy as np
import ufl
import basix

from mpi4py import MPI
from dolfinx import fem, io
import dolfinx.fem.petsc
from dolfinx.fem import Constant, dirichletbc, Function, functionspace, locate_dofs_topological
import dolfinx.mesh
from dolfinx.io import XDMFFile

Implentation des paramétres de la plaque étudié

In [42]:
L  = 0.301 # Longueur de la plaque
W_dim = 0.241 # Largeur de la plaque
N = 40 # Nombre de subdivision de la plaque
domain = dolfinx.mesh.create_rectangle(
    MPI.COMM_WORLD, [[0, 0], [L, W_dim]], [N, N] , cell_type=dolfinx.mesh.CellType.quadrilateral
) # Création du "mesh" de la plaque elle nous servira tous le long du programme pour appeler notre forme


# material parameters
thick = 0.00104 # épaisseur de la plaque
E = 188.08e9 # Module de Young du matériaus [MPa]
nu = 0.34 # Coef de poisson du matériau
rho = 7790.0 # Masse volumique [kg/L]

## Création des des matrices et des constantes :
- Matrice des Moments
$$
\mathbf{D} = D \begin{bmatrix}
1 & \nu & 0 \\
\nu & 1 & 0 \\
0 & 0 & \frac{1-\nu}{2}
\end{bmatrix} 
\quad \text{avec} \quad 
D = \frac{E \times h^3}{12(1-\nu^2)}
$$
- Constante des efforts de cisaillement
$$
F = \frac{5}{6} \times \frac{E \times h}{2(1 + \nu)}
$$
- Matrice de Masse
$$
\mathbf{M} = M \begin{bmatrix}
1 & 0 & 0 \\
0 & \frac{h^2}{12} & 0 \\
0 & 0 & \frac{h^2}{12}
\end{bmatrix}
\quad \text{avec} \quad
M = \rho h
$$

- Matrice des précontraintes

$$
\begin{aligned}
\underline{\underline{L}} &= \int_h \underline{\underline{\tilde{\sigma}}}(z) z^2 \, dz \\
\underline{\underline{P}} &= \int_h \underline{\underline{\tilde{\sigma}}}(z) \, dz
\end{aligned}
\qquad \text{donc} \qquad
\underset{\approx}{\Omega} = \begin{bmatrix}
\underline{\underline{P}} & 0 & 0 \\
0 & \underline{\underline{L}} & 0 \\
0 & 0 & \underline{\underline{L}}
\end{bmatrix}
$$

In [43]:
# bending stiffness
D = fem.Constant(domain, E * thick**3 / (1 - nu**2) / 12.0)
# shear stiffness
F = fem.Constant(domain, E / 2 / (1 + nu) * thick * 5.0 / 6.0)
# Inertilal constant
M = fem.Constant(domain , rho * thick)
# Acousto-Elastic Constants
sig0 = 10.0e6 # Pre-stress in the plate [Pa]
P = fem.Constant(domain, sig0 * thick)
L_m = fem.Constant(domain, sig0 * thick**3 / 12.0)

def curvature(u):
    (w, theta) = ufl.split(u)
    return ufl.as_vector(
        [theta[0].dx(0), theta[1].dx(1), theta[0].dx(1) + theta[1].dx(0)]
    )

def shear_strain(u):
    (w, theta) = ufl.split(u)
    return ufl.grad(w) - theta


def bending_moment(u):
    DD = ufl.as_matrix([[D, nu * D, 0], [nu * D, D, 0], [0, 0, D * (1 - nu) / 2.0]])
    return ufl.dot(DD, curvature(u))

def inertial(u) :
    MM = ufl.as_matrix([[M, 0, 0], [0, M*thick**2/12, 0], [0, 0, M*thick**2/12]])
    return ufl.dot(MM, u)


def shear_force(u):
    return F * shear_strain(u)


def residual_stress_shear(u, v) : 
    """Énergie de précontrainte (cisaillement)"""
    (dw, dtheta) = ufl.split(u)
    (w_, theta_) = ufl.split(v)
    
    return P * ufl.dot(ufl.grad(dw), ufl.grad(w_))

def residual_stress_bending(u, v):
    """Énergie de précontrainte (flexion)"""
    (dw, dtheta) = ufl.split(u)
    (w_, theta_) = ufl.split(v)
    
    return L_m * ufl.inner(ufl.grad(dtheta), ufl.grad(theta_))

On crée un vecteur avec 1 déplacement vertical et 2 rotation autour de la courbure :

$U_e = U_3$

$T_e = \begin{bmatrix} \phi_1  \\ \phi_2 \end{bmatrix} $

In [44]:

Ue = basix.ufl.element("P", domain.basix_cell(), 2)
Te = basix.ufl.element("P", domain.basix_cell(), 1, shape=(2,))
V = fem.functionspace(domain, basix.ufl.mixed_element([Ue, Te]))

Création des fonctions pour la formulation faible 

In [45]:
u = fem.Function(V, name="Unknown")
u_ = ufl.TestFunction(V)
(w_, theta_) = ufl.split(u_)
du = ufl.TrialFunction(V)
(dw , dtheta) = ufl.split(du)

Définition de la formulation Faible

In [46]:
dx = ufl.Measure("dx", domain=domain)
L = fem.Constant(domain, 0.0) * w_ * dx

# On définit la forme de raideur K complète
a = (
    ufl.inner(bending_moment(du), curvature(u_))
    + ufl.inner(shear_force(du), shear_strain(u_))
    + residual_stress_bending(du, u_) 
    + residual_stress_shear(du, u_)
) * dx

Définition des conditions aux limites de la plaque

In [47]:
def border(x):
    return np.logical_or(np.isclose(x[0], 0), np.isclose(x[0], 0))

facet_dim = 1
clamped_facets = dolfinx.mesh.locate_entities_boundary(domain, facet_dim, border)
clamped_dofs = fem.locate_dofs_topological(V, facet_dim, clamped_facets)

u0 = fem.Function(V)
bcs = []

Résolution du probléme Linéaire : Optionnel

In [48]:
from dolfinx.fem.petsc import LinearProblem

problem = LinearProblem(
    a, L, u=u, bcs=bcs, petsc_options={"ksp_type": "preonly", "pc_type": "lu"}, petsc_options_prefix="Plate"
)
problem.solve()

with io.VTKFile(domain.comm, "plates.xdmf", "w") as vtk:
    w = u.sub(0).collapse()
    w.name = "Deflection"
    vtk.write_function(w)

Affichage de la solution du probléme Linéaire

In [49]:
import pyvista
from dolfinx import plot

pyvista.set_jupyter_backend('trame')

Vw = w.function_space
w_topology, w_cell_types, w_geometry = plot.vtk_mesh(Vw)
w_grid = pyvista.UnstructuredGrid(w_topology, w_cell_types, w_geometry)
w_grid.point_data["Deflection"] = w.x.array
w_grid.set_active_scalars("Deflection")
warped = w_grid.warp_by_scalar("Deflection", factor=5)

plotter = pyvista.Plotter()
plotter.add_mesh(
    warped,
    show_scalar_bar=True,
    scalars="Deflection",
)
edges = warped.extract_all_edges()
plotter.add_mesh(edges, color="k", line_width=1)
plotter.show()

theta = u.sub(1).collapse()
Vt = theta.function_space
theta_topology, theta_cell_types, theta_geometry = plot.vtk_mesh(Vt)
theta_grid = pyvista.UnstructuredGrid(theta_topology, theta_cell_types, theta_geometry)
beta_3D = np.zeros((theta_geometry.shape[0], 3))
beta_3D[:, :2] = theta.x.array.reshape(-1, 2) @ np.array([[0, -1], [1, 0]])
theta_grid["beta"] = beta_3D
theta_grid.set_active_vectors("beta")
"""
"""
plotter = pyvista.Plotter()
plotter.add_mesh(
    theta_grid.arrows, lighting=False, scalar_bar_args={"title": "Rotation Magnitude"}
)
plotter.add_mesh(theta_grid, color="grey", ambient=0.6, opacity=0.5, show_edges=True)
plotter.show()

Widget(value='<iframe src="http://localhost:36577/index.html?ui=P_0x7135b87aaad0_10&reconnect=auto" class="pyv…

Widget(value='<iframe src="http://localhost:36577/index.html?ui=P_0x71358818fb10_11&reconnect=auto" class="pyv…

Analyse Modale du probléme 
-------
Dans cette partie nous allons résoudre l'équation :

$ KV - \lambda MV = 0 $

On récupérera les fréquences propres gràce à $ \lambda = \omega ^2 $ et $ \omega = 2 \pi f $

Définition de la Matrice de masse en formulation faible et assemblage des matrices de Masse et de raideur

In [50]:
m_form =  ufl.dot(inertial(u_), du) * dx

K = fem.petsc.assemble_matrix(fem.form(a), bcs)
K.assemble()
M = fem.petsc.assemble_matrix(fem.form(m_form), bcs)
M.assemble()

Solver du probléme aux modes propres

In [51]:
from petsc4py import PETSc
from slepc4py import SLEPc

N_eig = 50
eigensolver = SLEPc.EPS().create(MPI.COMM_WORLD)
eigensolver.setDimensions(N_eig)
eigensolver.setProblemType(SLEPc.EPS.ProblemType.GHEP)
st = SLEPc.ST().create(MPI.COMM_WORLD)
st.setType(SLEPc.ST.Type.SINVERT)
st.setShift(0.1)
st.setFromOptions()
eigensolver.setST(st)
eigensolver.setOperators(K, M)
eigensolver.setFromOptions()

# Compute eigenvalue-eigenvector pairs
eigensolver.solve()
evs = eigensolver.getConverged()
vr, vi = K.getVecs()
u_output = Function(V)
u_output.name = "Eigenvector"
evs = eigensolver.getConverged()
print(f"Nombre de modes convergés : {evs}")

Nombre de modes convergés : 53


Trouvé l'ensemble des solutions :

In [52]:
import pyvista
import numpy as np

# Choix du mode à afficher (ex: 3, 4, 5... pour sauter les modes de corps rigide)
mode_cible = 7

# Vérification que le mode demandé a bien été calculé
if mode_cible < evs:
    # 1. Récupération de la valeur propre et calcul de la fréquence
    l = eigensolver.getEigenpair(mode_cible, vr, vi)
    freq = np.sqrt(max(0, l.real)) / (2 * np.pi)
    
    # 2. Extraction sécurisée de la déflexion w
    u_output.x.array[:] = vr.getArray()
    u_output.x.scatter_forward()
    w_mode = u_output.sub(0).collapse()
    
    # 3. Création du maillage PyVista
    topology, cell_types, geometry = plot.vtk_mesh(w_mode.function_space)
    grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)
    
    # Normalisation pour l'affichage (max amplitude = 1)
    w_array = w_mode.x.array
    grid.point_data["Amplitude"] = w_array / np.max(np.abs(w_array))
    warped = grid.warp_by_scalar("Amplitude", factor=0.05)
    
    # 4. Configuration et affichage de la fenêtre unique
    plotter = pyvista.Plotter(window_size=[1000, 700])
    plotter.set_background("white")
    plotter.add_text(
                    f"Mode {mode_cible} : {freq:.1f} Hz", 
                    position="lower_edge", 
                    font_size=14, 
                    color="black"
                    )
    plotter.add_mesh(warped, cmap="jet", show_scalar_bar=False , lighting=False)
    plotter.add_mesh(warped.extract_all_edges(), color="black", opacity=0.1)
    
    plotter.enable_parallel_projection()
    plotter.show()
else:
    print(f"Le mode {mode_cible} n'a pas été calculé. Modes convergés : {evs}")

Widget(value='<iframe src="http://localhost:36577/index.html?ui=P_0x713607fdd150_12&reconnect=auto" class="pyv…

Affichage des différents mode :
----


In [53]:
import numpy as np
# Vérification que le mode demandé a bien été calculé

mode = np.linspace(3, evs, evs, dtype=int)
freq_ha = []

# 1. Récupération de la valeur propre et calcul de la fréquence
for m in mode[:-2] :
    l = eigensolver.getEigenpair(m, vr, vi)
    freq = np.sqrt(max(0, l.real)) / (2 * np.pi)
    freq_ha.append(freq)

print(freq_ha)

[np.float64(108.94172791142965), np.float64(108.94172791142965), np.float64(141.92726464033285), np.float64(184.35499252952144), np.float64(187.95557364361164), np.float64(216.07905043603824), np.float64(257.1511374524632), np.float64(296.78331648896045), np.float64(299.07256144737124), np.float64(358.1960573781606), np.float64(388.04631045691497), np.float64(418.4753045972489), np.float64(420.8439627818132), np.float64(457.5249092833722), np.float64(479.179142149845), np.float64(587.4027825329023), np.float64(608.5031673215552), np.float64(611.4153644844976), np.float64(624.3205703068284), np.float64(645.6423974117315), np.float64(676.7842304913135), np.float64(727.2509319559359), np.float64(789.6653108671222), np.float64(808.5210200707786), np.float64(871.5959197087682), np.float64(904.757287317204), np.float64(931.9287818872483), np.float64(931.9287818872483), np.float64(957.2093461870779), np.float64(968.7380524229346), np.float64(1020.4647590474729), np.float64(1057.528877502869),